In [1]:
import matplotlib.pyplot as plt

import finesse

finesse.plotting.init() # initialise matplotlib rcParams to appropriate values for a "display" mode

ModuleNotFoundError: No module named 'finesse'

## Finesse 3 Introduction --- Fabry-Perot Cavity

This notebook is intended as an introduction to the Finesse 3 syntax and features by way of a simple example of a Fabry-Perot cavity configuration.

Full documentation for Finesse 3 can be found at: https://finesse.ifosim.org/docs/latest/index.html.

Finesse 3 is in alpha stage of development. You can find the documentation, executables and code for the stable Finesse 2 at http://www.gwoptics.org/finesse.

**Key Points**

- **Building models**
Before you can perform simulations of your optical system you must first build a model of it. Finesse offers two ways in which you can build a model. Firstly using KatScript, which is a declarative domain specific language developed for building models of interferometers. This way of building removes the complexity of dealing with Python and keeps you focused on the physics of your model.
- **Model** can be placed in a separate `.kat` file or saved as a string assigned to a variable into a python script. KatScript in Finesse 3 works largely with the same syntax as Finesse 2.
- This model object stores the [network](https://finesse.readthedocs.io/en/latest/api/generated/finesse.model.Model.network.html#finesse.model.Model.network) which is a directed graph
  where the `node_type` is the [full_name](https://finesse.readthedocs.io/en/latest/api/components/node/generated/finesse.components.node.Node.full_name.html#finesse.components.node.Node.full_name) property
  of a Node object. See [this documentation](https://finesse.readthedocs.io/en/latest/usage/advanced_usage/nodesystem/index.html) for in-depth details on the port and node system of Finesse 3.
- **Parameter** references can be set up in `.kat` files by using the `$` syntax:
  - track the parameter of another component using `$<comp_name>.param` -- e.g: `ad ampdet $EOM.f n1` tells Finesse to
    create an amplitude detector where the detection frequency is set up to track the frequency of the modulator named `EOM`.
  - more complicated referencing / tracking can be performed using `$$<eqn>$$` -- e.g. `ad ampdef $$EOM1.f + 2*EOM2.f$$ n1` tells Finesse
    to create an amplitude detector where the detection frequency is set to track a frequency equal to EOM1 modulation freq. plus two times EOM2 modulation freq.
- **Simulations** can be run over ranges of parameters with the `xaxis`, `x2axis` functions. Building of the Simulation object is performed internally within
  these functions using the model associated with the tunable parameters.

In the following cell we set up a Finesse model of a cavity using the `.kat` style syntax. The amplitude detectors `Cp1` and `Cm1` are initialised such that the frequencies of the field that they detect track the positive and negative values of the modulator `EOM1`, respectively. This means that if `EOM1.f` is modified in any way later on, then `Cp1` and `Cm1` will still detect the upper and lower sidebands of the carrier circulating in the cavity. 

**Optical setup**

Laser L0 -> space s0 -> modulator EOM1 -> space s1 -> mirror ITM -> space sCAV -> mirror ETM

![alt text](FPcavity.png "Fabry-Perot cavity")

**Optical detectors**

Refl -- field entering the cavity

Cav -- field inside the cavity

Cavp1 -- modulated field inside the cavity (positive sideband)

Cavm1 -- modulated field inside the cavity (negative sideband)

Trans -- field transmitted by the cavity

In [ ]:
kat = finesse.Model()
kat.parse("""
l L0 P=1

s s0 L0.p1 EOM1.p1
mod EOM1 f=100M midx=0.1 order=1 #mod_type=pm
s s1 EOM1.p2 ITM.p1 L=1

m ITM R=0.99 T=0.01
s sCAV ITM.p2 ETM.p1 L=1
m ETM R=0.99 T=0.01

ad Refl ITM.p1.o f=0
ad Cav ETM.p1.i f=0
ad Cavp1 ETM.p1.i f=EOM1.f
ad Cavm1 ETM.p1.i f=(-EOM1.f)
ad Trans ETM.p2.o f=0

# Scan over the detuning DOF of m1 from -180 deg to +180 deg with 400 points.
xaxis(ITM.phi, lin, -100, 280, 1000)
""")

# run a simulation where we vary the tuning of m1 from -100 deg to 280 deg with 1000 steps
out = kat.run()
out.plot(logy=True, figsize_scale=1.5);

Upon running the simulation, we obtain an object `out` which acts as a wrapper around a `numpy` structured array whose named elements are the names of outputs in a model. So we can access, e.g., the output of the detector `Cp1` with `out['Cp1']`; whilst `out.x1` gives the data of the axis over which the first parameter was varied.

## More details on Finesse and our simulation

There are various coordinate systems involved in modelling the mirror component. Each mechanical and optical node has an associated coordinate system. Shown in the figure below are the coordinate systems for the optical p1 (Port 1) and the mechanical mode mech.

![alt text](Finesse3Mcoordinates.png "Title")

The optical nodes, as for all optical nodes for any component, are in a left-handed coordinate system, shown in blue. The incoming and output going nodes represent the beam in the incident plane. For a mirror the angle of incidence is fixed to zero, α=0α=0. A beamsplitter component is exactly the same as a mirror component, except that the angle of incidence can be non-zero. The outgoing node is a reflection of the incoming in the z and x coordinates.

The mechanical node coordiante system is shown in black and is a right-handed coordinate system. A positive z motion is the surface normal of mirror surface going in the port 1 direction. Yaw and pitch are right-handed rotations around the y and x axes respectively.